# Data exploration: [AMES dataset](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

In this notebook, we will explore a dataset that we will use again later on during the training to train models.

This dataset is a good playground because it contains all kinds of variables, has missing values, many features, and those features aren't necessarily suited to the methods we try to apply to them: it's a good model of the datasets you'll meet day to day doing data science.

Today, we will follow the following steps:

1. state the problem
2. obtain the data
3. clean and encode the data
4. explore the data
5. apply what the exploration taught us

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import scipy.stats
import seaborn

pandas.set_option("future.no_silent_downcasting", True)

## Problem statement

The end goal for our study is to be able to predict prices for the housing market in Iowa, in the US.

In order to have a correctly stated problem, it's required to formulate from the get go what are the expected outputs and how we will evaluate them.

- *Define the model output given the end goal.*
- *Suggest one or several ways to evaluate the quality of the models we will develop later on.*
- *What would be the issue if we delayed the definition of those output and evaluation elements?*

Answer in the next cell, as a comment.

In [ ]:
# Your answers here, as comments

### Additional hints

- The output: what do we want to predict, and is it a class or a continuous value?
- The metrics: which regression errors do you know? Does an error of $20,000 weigh as much on a $100,000 house as on a $700,000 one?
- The issue: what happens if we pick the metric after seeing the results?

### Solution

- The model output is the house's sale price (`SalePrice`, in dollars): it's a regression problem.
- Metrics: the mean absolute error (MAE), in dollars, easy to read; the root mean squared error (RMSE), which penalizes large errors more; and a relative error, since an error of $20,000 doesn't weigh the same on a $100,000 house and on a $700,000 one: the RMSE computed on the logarithm of the price, the metric of the Kaggle competition this data comes from.
- Without defining them up front, we introduce a statistical bias: we could pick, after the fact, the metrics that maximize our results.

## Fetching the Data

The next cell downloads the data into the `ames/` directory. It contains in particular a `train.csv` file you can use as training and validation set, and a `test.csv` file that simulates production use: the targets aren't available. The CSV's variables are described in the `ames/data_description.txt` file, which the cell also downloads to your machine on Colab.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
!ls -l ames/
try:
  from google.colab import files

  files.download("ames/data_description.txt")
except ImportError:  # Outside Colab: the file is in the explorer, under ames/
  pass

### Reading the data

- *Read `ames/train.csv` and `ames/test.csv` into the `train_df` and `test_df` DataFrames with `pandas.read_csv`. The index column of the data is named `Id` in the CSV.*
- *Display the names of the columns, along with the dimensions of the data (number of rows, number of columns).*
- *Display the beginning of `ames/data_description.txt`.*

**Useful documentation**

- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [`pandas.DataFrame.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html)

In [ ]:
# Your code here

#### Additional hints

- `pandas.read_csv(path, index_col="Id")`: the `Id` column becomes the index instead of a variable.
- `df.columns`: the names of the columns; `df.shape`: the (rows, columns) pair.
- `f.read()[:2000]`: the first 2,000 characters of a file opened with `open`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
train_df = pandas.read_csv(..., index_col=...)
test_df = ...

print(f"Columns: {', '.join(train_df.columns)}")
print(f"Shape of train_df: {...}")
print(f"Shape of test_df: {...}")

with open("ames/data_description.txt") as f:
  print(f.read()[...])
```

#### Solution

In [ ]:
train_df = pandas.read_csv("ames/train.csv", index_col="Id")
test_df = pandas.read_csv("ames/test.csv", index_col="Id")

print(f"Columns: {', '.join(train_df.columns)}")
print(f"Shape of train_df: {train_df.shape}")
print(f"Shape of test_df: {test_df.shape}")

# The beginning of the variables' description
with open("ames/data_description.txt") as f:
  print(f.read()[:2000])

# A preview of the first rows
train_df.head()

## Extracting variables

*Create the following variables:*

- *`train_X` that contains all the columns of `train_df` except `SalePrice`*
- *`train_y` that contains the `SalePrice` column of `train_df`*
- *`test_X` that contains all the columns of `test_df`*

*Also create the `all_X` variable, the concatenation of `train_X` and `test_X`, that will allow us to preprocess everything the same way.*

*__Warning, using `test_X` (and therefore `all_X`) can introduce a statistical bias. You should only use `train_X` when using the data to compute a value!__*

The course recommends setting data aside without looking at it; why touch `test_X` then? In this data's setting (a Kaggle competition), the test set has no target: it's only used to produce the predictions. We only look at it to know what we'll have to handle (its missing values), every filling value is computed on `train_X`, and we encode both tables together so that they have the same columns: a category that only exists in the test set becomes a column there, which is 0 on the whole training set.

**Useful documentation**

- [`pandas.DataFrame.drop`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Your code here

### Additional hints

- `train_df.drop(columns="SalePrice")`: a copy without the column.
- `train_df["SalePrice"]`: a column, as a `Series`.
- `pandas.concat([a, b])`: stacks the rows of `a` and `b`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
train_X = train_df.drop(...)
train_y = ...
test_X = ...
all_X = pandas.concat(...)

print(train_X.shape, train_y.shape, test_X.shape, all_X.shape)
```

### Solution

In [ ]:
train_X = train_df.drop(columns="SalePrice")
train_y = train_df["SalePrice"]
test_X = test_df
all_X = pandas.concat([train_X, test_X])

print("Shape of the training data:", train_X.shape, train_y.shape)
print("Shape of the test data:", test_X.shape)
print("Concatenation:", all_X.shape)

## Data Cleaning

This step, together with exploration, involves (at least) 4 steps:

- handling missing values
- preprocessing (text, image, …)
- standardization
- transformation

Here, we won't need specific preprocessing, since the data is mainly categorical or continuous, without text or image data.

### Missing Values

Format differences, data gathering conditions and many other factors lead to many missing values in most datasets.

In this dataset, `NA` often means absence (no pool, no garage): read `data_description.txt` to tell an absence, which becomes a category, from a real missing value, which has to be imputed.

It's often useful to quantify what's missing and to make up for it when it makes sense. For instance, note that to use `sklearn`, no value can be missing for almost all models to work.

- *Find the total number of missing elements in `train_X`, then in `test_X`.*
- *Compute, for both tables, the percentage of missing elements per column, keeping only the columns that miss some, sorted to display first those that miss the most. Name the results `missing_train` and `missing_test`.*
- *Display all the columns containing at least one missing value (train and test sets combined), then those that only have some in the test set.*

**Useful documentation**

- [`pandas.DataFrame.isnull`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html)
- [`pandas.DataFrame.sum`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)
- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)

In [ ]:
# Your code here

#### Additional hints

- `df.isnull()`: a table of booleans, `True` where a value is missing.
- `.sum()` counts the `True` values of each column, `.sum().sum()` those of the whole table; `.mean() * 100` gives the percentage per column.
- `s[s > 0]` keeps the positive elements of a `Series`; `s.sort_values(ascending=False)` sorts it in decreasing order.
- `set(missing_train.index)`: the set of the training set's columns with missing values; `|` makes their union, `-` their difference.
- Expected: 7,829 missing values in `train_X`, 7,878 in `test_X`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def columns_with_missing_values(df: pandas.DataFrame) -> pandas.Series:
  percentages = ...  # percentage of missing values per column
  percentages = percentages[...]
  return percentages.sort_values(...)


print("Missing values in train_X:", train_X.isnull().sum().sum())
print("Missing values in test_X:", ...)

missing_train = columns_with_missing_values(train_X)
missing_test = ...
print(missing_train)
print(missing_test)

print(set(missing_train.index) | ...)
print(...)  # only in the test set
```

#### Solution

In [ ]:
def columns_with_missing_values(df: pandas.DataFrame) -> pandas.Series:
  df_na: pandas.Series = df.isnull().mean() * 100
  df_na = df_na[df_na > 0]
  return df_na.sort_values(ascending=False)


def display_percentages_series(s: pandas.Series) -> None:
  print("\n".join([f"{name:>20.20} {value:5.2f}%" for name, value in s.items()]))


print("Number of missing values in the training set:", train_X.isnull().sum().sum())
print("Number of missing values in the test set:", test_X.isnull().sum().sum())
print()

print("Percentage of missing values per feature in the training set:")
missing_train = columns_with_missing_values(train_X)
display_percentages_series(missing_train)
print()

print("Percentage of missing values per feature in the test set:")
missing_test = columns_with_missing_values(test_X)
display_percentages_series(missing_test)
print()

print("Columns with missing values in the training set or the test set:")
print(", ".join(sorted(set(missing_train.index) | set(missing_test.index))))
print()

print("Columns with missing values only in the test set:")
print(", ".join(sorted(set(missing_test.index) - set(missing_train.index))))

### Handling Missing Data

There's no universal approach to filling missing values. Each variable must be handled case by case.

*Based on the documentation in the `ames/data_description.txt` file, split the columns with missing values into groups, one per filling method, and fill them with `DataFrame.fillna`:*

- *`cols_mean`: the mean (or the median) for a real variable whose value is really missing;*
- *`cols_mode`: the mode (the most frequent value) for a categorical variable whose value is really missing;*
- *`cols_zero`: 0 for a quantity that is missing because the element is absent (the area of a basement that doesn't exist);*
- *a value the documentation dictates: `Functional` is `Typ` unless deductions are warranted;*
- *all the others: the `"NA"` category, which means absence (no alley, no pool).*

*The filling values (mean, mode) are computed on `train_X` only. In the end, `all_X` must have no missing value left.*

**Useful documentation**

- [`pandas.DataFrame.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)
- [`pandas.DataFrame.mean`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)
- [`pandas.DataFrame.mode`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mode.html)

In [ ]:
# Your code here

#### Additional hints

- `all_X.fillna(train_X[cols_mean].mean())`: passing a `Series` (column name → value) fills each column with its own value, computed here on the training set.
- `train_X[cols_mode].mode().iloc[0]`: `mode()` returns a table (a column can have several modes); its first row keeps one per column.
- The columns that only have missing values in the test set (`MSZoning`, `Utilities`, `KitchenQual`…) miss one or two values: real missing values.
- The basement and garage areas and counts (`BsmtFinSF1`, `GarageCars`…) are missing when there's no basement or garage: 0.
- `all_X.fillna("NA")`, last, fills everything else.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
cols_mean = ["LotFrontage", ...]
all_X = all_X.fillna(train_X[cols_mean].mean())

cols_mode = ["MSZoning", ...]
all_X = all_X.fillna(train_X[cols_mode].mode().iloc[0])

cols_zero = ["GarageArea", ...]
all_X[cols_zero] = all_X[cols_zero].fillna(...)

all_X["Functional"] = ...
all_X = ...

print(all_X.isnull().sum().sum())  # 0 expected
```

#### Solution

In [ ]:
# Filling with the mean
cols_mean = ["LotFrontage", "GarageYrBlt"]
all_X = all_X.fillna(train_X[cols_mean].mean())

# Filling with the mode
cols_mode = [
  "MSZoning",
  "Electrical",
  "KitchenQual",
  "Exterior1st",
  "Exterior2nd",
  "SaleType",
  "Utilities",
]
all_X = all_X.fillna(train_X[cols_mode].mode().iloc[0])

# Filling with 0: no basement, garage or veneer
cols_zero = [
  "GarageArea",
  "GarageCars",
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtFullBath",
  "BsmtHalfBath",
  "BsmtUnfSF",
  "MasVnrArea",
  "TotalBsmtSF",
]
all_X[cols_zero] = all_X[cols_zero].fillna(0)

# A specific filling, dictated by data_description.txt
all_X["Functional"] = all_X["Functional"].fillna("Typ")

# All the other NAs get the NA string value, which will be a category
all_X = all_X.fillna("NA")

# Check that we didn't forget anything
print(all_X.isnull().sum().sum())

### Categorical Variables Encoded as Numbers

Some categories, such as `MSSubClass`, are encoded with numerical codes expressing categories (see `data_description.txt`). If we do nothing, this variable will be treated as numerical, even though the categories it describes aren't even ordinal.

In [ ]:
# To achieve that, we just have to cast the feature as a string column
cols_numerical2label = ["MSSubClass"]
all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

### Ordinal variables

In order not to lose the order information of our ordinal variables, we will have to use label encoding as seen during the presentation: each level becomes an integer, in the levels' order.

- *Use label encoding for the `BsmtCond` and `FireplaceQu` variables (from `NA`, absent, to `Ex`, excellent: see `data_description.txt`) with `DataFrame.replace`, then convert them to integers.*

The solution handles all the dataset's ordinal variables the same way: what follows starts from its result.

**Useful documentation**

- [`pandas.DataFrame.replace`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)
- [`pandas.DataFrame.astype`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

In [ ]:
# Your code here

#### Additional hints

- `all_X.replace({"BsmtCond": {"NA": 0, "Po": 1}})`: one replacement dictionary per column, under the column's name.
- Both variables have the same quality scale: `NA`, `Po`, `Fa`, `TA`, `Gd`, `Ex`.
- `all_X[columns].astype(int)`: the conversion to integers, once the strings are replaced.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
quality_mapping = dict(NA=0, Po=1, Fa=..., TA=..., Gd=..., Ex=...)
replace_mapping = dict(BsmtCond=quality_mapping, FireplaceQu=...)
all_X = all_X.replace(replace_mapping)

ordinal_columns = list(replace_mapping)
all_X[ordinal_columns] = all_X[ordinal_columns].astype(...)
```

#### Solution

In [ ]:
# Creation of the dictionary of dictionaries used to replace the ordinal variables by integers
# Key   : "Name of the variable"
# Value :  Replacement dictionary of the variable's levels
replace_mapping = dict(
  Alley=dict(NA=0, Grvl=1, Pave=2),
  BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
  BsmtFinType1=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  BsmtFinType2=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
  LandSlope=dict(Sev=1, Mod=2, Gtl=3),
  LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
  PavedDrive=dict(NA=0, N=1, P=2, Y=3),
  Street=dict(Grvl=1, Pave=2),
  Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  BsmtCond=dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5),
)
# All the following variables can use the same replacement dictionary
quality_columns = [
  "BsmtCond",
  "BsmtQual",
  "ExterCond",
  "ExterQual",
  "FireplaceQu",
  "GarageCond",
  "GarageQual",
  "HeatingQC",
  "KitchenQual",
  "PoolQC",
]
quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
for quality_column in quality_columns:
  replace_mapping[quality_column] = quality_mapping

all_X = all_X.replace(replace_mapping)

# Explicit cast of these variables to integers
ordinal_columns = list(replace_mapping.keys())
all_X[ordinal_columns] = all_X[ordinal_columns].astype(int)

### One hot encoding for nominal variables

Now that the ordinal variables are handled, we still have to handle the categorical (nominal) variables: `pandas.get_dummies`'s one-hot encoding replaces each one with one indicator column per level.

- *Before calling `pandas.get_dummies`, predict the number of columns `all_X` will have: the numerical columns stay, and each non-numerical column is replaced by as many columns as it has levels (`nunique`).*
- *Apply the transformation to `all_X`, then check your prediction.*

**Useful documentation**

- [`pandas.DataFrame.select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html)
- [`pandas.DataFrame.nunique`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html)
- [`pandas.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)

In [ ]:
# Your code here

#### Additional hints

- `all_X.select_dtypes("number")`: the numerical columns; `select_dtypes(exclude="number")`: the others.
- `df.nunique()`: the number of levels of each column; `.sum()` adds them up.
- `df.shape[1]`: the number of columns.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
numerical = all_X.select_dtypes(...)
categorical = all_X.select_dtypes(exclude=...)
print("Predicted columns:", numerical.shape[1] + ...)

all_X = pandas.get_dummies(...)
print("Columns obtained:", ...)
```

#### Solution

In [ ]:
numerical = all_X.select_dtypes("number")
categorical = all_X.select_dtypes(exclude="number")
print("Shape before:", all_X.shape)
print("Numerical columns:", numerical.shape[1])
print("Non-numerical columns:", categorical.shape[1])
print("Predicted columns:", numerical.shape[1] + categorical.nunique().sum())

all_X = pandas.get_dummies(all_X)
print("Shape after:", all_X.shape)

Each level of each categorical variable was turned into a feature. The input matrix went from its original size of $\mathbb{R}^{2919 \times 79}$ to $\mathbb{R}^{2919 \times 244}$.

### Reassembling

Now that the cleaning process is over, we split `all_X` back into `train_X` and `test_X`, using their indexes (the houses' `Id`s).

In [ ]:
train_X = all_X.loc[train_X.index]
test_X = all_X.loc[test_X.index]
print(train_X.shape, test_X.shape)

### Checking the prepared data

- *Check that `train_X` and `test_X` have the same columns, and that no value is missing in them.*
- *List the indicator columns (those `get_dummies` created) that are 0 on the whole training set. Where do they come from, and what can a model trained on the training set learn from them?*

**Useful documentation**

- [`pandas.Index.equals`](https://pandas.pydata.org/docs/reference/api/pandas.Index.equals.html)
- [`pandas.DataFrame.any`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html)

In [ ]:
# Your code here

#### Additional hints

- `train_X.columns.equals(test_X.columns)`: `True` if both tables have the same columns, in the same order.
- `get_dummies`'s columns are booleans: `train_X.select_dtypes(bool)`.
- `df.any()` tells, for each column, whether it holds at least one `True`; `~` negates it.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
print("Same columns:", train_X.columns.equals(...))
print("Missing values:", ..., ...)

indicators = train_X.select_dtypes(bool)
print(list(indicators.columns[...]))
```

#### Solution

In [ ]:
print("Same columns:", train_X.columns.equals(test_X.columns))
print("Missing values:", train_X.isnull().sum().sum(), test_X.isnull().sum().sum())

indicators = train_X.select_dtypes(bool)
print("Indicators at 0 on the whole training set:")
print(list(indicators.columns[~indicators.any()]))

Both tables have the same columns and no missing value. Only one indicator is 0 on the whole training set: `MSSubClass_150`, a level that only appears in the test set. It only exists because we encoded both tables together; constant on the training set, it teaches a model nothing, and the model will treat these test houses as if they had none of the levels it saw. In production, an unknown level will come the same way, without a chance to re-encode: scikit-learn's `OneHotEncoder(handle_unknown="ignore")`, fitted on the training set only, encodes it as zeros.

## Data exploration

### Target variable analysis

Let's start by analyzing the most important variable: the target variable. This [seaborn doc page](https://seaborn.pydata.org/tutorial/distributions.html#plotting-univariate-histograms) shows the different ways to explore such a variable.

- *Use `seaborn` to visualize the distribution of the output variable: a histogram, with a density estimate.*
- *How are the prices distributed?*

**Useful documentation**

- [`seaborn.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html)

In [ ]:
# Your code here

#### Additional hints

- `seaborn.histplot(train_y, kde=True)`: the histogram, and the curve of a kernel density estimate (KDE).
- `plt.show()` displays the figure.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
seaborn.histplot(..., kde=...)
plt.show()
```

#### Solution

In [ ]:
seaborn.histplot(train_y, kde=True)
plt.show()

The distribution is skewed: most sales are between $100,000 and $250,000, and a long tail stretches towards high prices, up to $755,000.

### Skewness of the output variable

A very skewed output variable hinders many methods: the rare high prices weigh a lot in a squared error, and an error of $20,000 counts as much on a cheap house as on an expensive one. The course suggests transforming such a variable (Box-Cox); for a price, the usual transformation is the logarithm.

- *Measure the skewness of `train_y` with `Series.skew`.*
- *Apply `numpy.log1p` to `train_y`, and measure the skewness again.*
- *Compare the histogram and the quantile-quantile plot (QQ-plot, against a normal distribution) before and after the transformation.*

**Useful documentation**

- [`pandas.Series.skew`](https://pandas.pydata.org/docs/reference/api/pandas.Series.skew.html)
- [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)
- [`scipy.stats.probplot`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html)

In [ ]:
# Your code here

#### Additional hints

- `train_y.skew()`: 0 for a symmetric distribution, positive for a tail to the right. Expected: about 1.88 before the transformation, 0.12 after.
- `numpy.log1p(train_y)`: $\log(1 + y)$, defined even for $y = 0$.
- `_, axes = plt.subplots(2, 2, figsize=(10, 8))`: a 2 × 2 grid of plots, `axes[row][column]`.
- `seaborn.histplot(y, kde=True, ax=ax)` draws into the `ax` plot.
- `scipy.stats.probplot(y, dist="norm", plot=ax)` plots the quantiles of `y` against those of a normal distribution: a normal distribution gives a straight line.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
train_y_log = numpy.log1p(...)
print("Skewness before:", train_y.skew())
print("Skewness after:", ...)

_, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, y in zip(axes, [train_y, train_y_log]):
  seaborn.histplot(y, kde=True, ax=row[0])
  scipy.stats.probplot(..., dist="norm", plot=row[1])
plt.tight_layout()
plt.show()
```

#### Solution

In [ ]:
train_y_log = numpy.log1p(train_y)
print(f"Skewness before: {train_y.skew():.2f}")
print(f"Skewness after: {train_y_log.skew():.2f}")

_, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, y, title in zip(
  axes, [train_y, train_y_log], ["SalePrice", "log(1 + SalePrice)"]
):
  seaborn.histplot(y, kde=True, ax=row[0])
  row[0].set_title(title)
  scipy.stats.probplot(y, dist="norm", plot=row[1])
  row[1].set_title(f"QQ-plot of {title}")
plt.tight_layout()
plt.show()

The skewness goes from about 1.88 to 0.12: after the logarithm, the histogram is nearly symmetric, and the QQ-plot nearly a straight line, whereas before the high prices clearly strayed above it.

Models don't require a normal target: linear regression assumes (for its tests and confidence intervals) normal residuals, and decision trees assume nothing about the target. Predicting the logarithm of the price is mostly a way to make an error count in proportion to the price: the RMSE on the log is the relative error of the first part. A model trained on `train_y_log` predicts a log-price, which `numpy.expm1` brings back to dollars.

### Categorical variable and price

A categorical variable, such as the neighborhood (`Neighborhood`), is explored with two plots: the number of sales of each level (count plot), and the mean price per level (bar plot). After `get_dummies`, `train_X` no longer has a `Neighborhood` column: use `train_df`.

- *Draw the count plot of `Neighborhood`, and the bars of the mean price per neighborhood, sorted by mean price.*
- *Which neighborhoods are the most expensive? How much can you trust the mean price of a neighborhood with few sales?*

**Useful documentation**

- [`seaborn: bar plots`](https://seaborn.pydata.org/tutorial/categorical.html#bar-plots)
- [`seaborn.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html)
- [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html)

In [ ]:
# Your code here

#### Additional hints

- With 25 neighborhoods, horizontal bars keep the names readable: the categorical variable on `y`.
- `seaborn.countplot(data=train_df, y="Neighborhood")` counts the rows of each level.
- `seaborn.barplot(data=train_df, x="SalePrice", y="Neighborhood")` plots the mean price of each level, with its 95% confidence interval (the black line).
- `order=` takes the list of the levels in display order: `train_df.groupby("Neighborhood")["SalePrice"].mean().sort_values().index`.
- `plt.subplots(1, 2, figsize=(12, 8), sharey=True)`: two plots side by side, sharing the neighborhoods' axis.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
order = train_df.groupby(...)["SalePrice"].mean().sort_values().index

_, (ax_count, ax_price) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
seaborn.countplot(data=train_df, y=..., order=order, ax=ax_count)
seaborn.barplot(data=train_df, x=..., y="Neighborhood", order=..., ax=ax_price)
plt.tight_layout()
plt.show()
```

#### Solution

In [ ]:
order = train_df.groupby("Neighborhood")["SalePrice"].mean().sort_values().index

_, (ax_count, ax_price) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
seaborn.countplot(data=train_df, y="Neighborhood", order=order, ax=ax_count)
ax_count.set_title("Number of sales")
seaborn.barplot(
  data=train_df, x="SalePrice", y="Neighborhood", order=order, ax=ax_price
)
ax_price.set_title("Mean price")
plt.tight_layout()
plt.show()

NoRidge, NridgHt and StoneBr have the highest mean prices (more than $300,000), MeadowV, IDOTRR and BrDale the lowest (about $100,000): the neighborhood explains a good part of the price.

The neighborhoods are very unevenly represented: 225 sales in NAmes, 2 in Blueste. The confidence interval of the mean price (the black line) widens when the sales are few: the mean price of a rare neighborhood is unreliable, and a model will have few examples to learn its effect from.

### Analyzing the Correlations to the Output Variable

Now that we have a good idea of the shape of the output variable, let's analyze the variables most correlated with it.

- *Compute the correlation of each variable of `train_X` with the output variable, and sort them.*
- *Display with a heatmap the correlation matrix of the ten most correlated variables and the ten most anti-correlated ones, between themselves and with the output variable.*
- *Which variables are the most correlated with the price?*

**Useful documentation**

- [`pandas.DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html)
- [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
# Your code here

#### Additional hints

- `train_X.corrwith(train_y)`: the correlation of each column with `train_y`; `.dropna()` removes those of the constant columns (`NaN`).
- `s.sort_values(ascending=False)`; `pandas.concat([s[:10], s[-10:]])`: the first ten and the last ten values.
- `train_X[columns].corr()`: the correlation matrix of these columns; `pandas.concat([s, m], axis=1)` adds the `s` column to it.
- `seaborn.heatmap(m, vmin=-1, vmax=1, cmap="vlag")`: a color scale centered on 0.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
target_corrs = train_X.corrwith(...).dropna().sort_values(...)
top_target_corrs = pandas.concat([...]).rename("SalePrice")
feature_corrs = train_X[top_target_corrs.index].corr()
corrs = pandas.concat([top_target_corrs, feature_corrs], axis=1)

_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=..., vmax=..., cmap="vlag", ax=ax)
plt.show()
```

#### Solution

In [ ]:
target_corrs = (
  train_X
  # Correlations with the output variable
  .corrwith(train_y)
  # Removal of the NaNs created: they correspond to the constant
  # columns
  .dropna()
  # Sort by correlation value: the most correlated columns
  # will be first, the most anti-correlated last
  .sort_values(ascending=False)
)

# The first ten and last ten values
top_target_corrs = pandas.concat([target_corrs[:10], target_corrs[-10:]])
top_target_corrs = top_target_corrs.rename("SalePrice")

# Correlations of the most correlated (and anti-correlated) variables between themselves
feature_corrs = train_X.loc[:, top_target_corrs.index].corr()

# Grouping the correlations with the target and the correlations between
# variables
corrs = pandas.concat([top_target_corrs, feature_corrs], axis=1)

# Display of the correlation matrix with a heatmap
_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=-1, vmax=1, cmap=seaborn.diverging_palette(220, 20, n=100))
ax.set_title("Largest correlations and anti-correlations")
plt.show()

print(top_target_corrs.round(2))

The overall quality (`OverallQual`, 0.79) and the living area (`GrLivArea`, 0.71) are the most correlated with the price, followed by other quality variables (exterior, kitchen, basement) and size variables (garage, basement, first floor). Several of these variables are also highly correlated with each other (`GarageCars` and `GarageArea`, `TotalBsmtSF` and `1stFlrSF`): they partly carry the same information. The anti-correlations, weaker, point to more modest houses (unfinished or detached garage, cinder block foundation).

### Exploring the Most Correlated Variables

We now know the most correlated variables. We still need to learn more about them.

- *Plot the joint distribution of `GrLivArea` and `SalePrice`, as well as the distribution of `SalePrice` for each value of `OverallQual`.*
- *What do you notice on those two plots?*

**Useful documentation**

- [`seaborn.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html)
- [`seaborn.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html)

In [ ]:
# Your code here

#### Additional hints

- `seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)`: the scatter plot of both variables, with the distribution of each on the edges.
- `seaborn.violinplot(x=train_X["OverallQual"], y=train_y)`: one violin (the density of `SalePrice`) per value of `OverallQual`.
- Look at the points that stray from the cloud, and at what `train_df` says about these sales (`SaleCondition`).

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
seaborn.jointplot(x=..., y=train_y)
plt.show()
seaborn.violinplot(x=..., y=train_y)
plt.show()
```

#### Solution

In [ ]:
seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)
plt.show()
seaborn.violinplot(x=train_X["OverallQual"], y=train_y)
plt.show()

print(
  train_df.loc[
    train_df["GrLivArea"] > 4000, ["GrLivArea", "SalePrice", "SaleCondition"]
  ]
)

The price grows with the living area and with the overall quality, and so does its spread: quality 9 and 10 houses have much more varied prices than quality 4 ones.

Two sales clearly stray from the cloud: the two largest areas (more than 4,500 sq ft), sold for less than $200,000. Are they errors in the dataset, or houses with unusual characteristics? `train_df` gives their sale condition: `Partial`, a house not yet completed when it was sold. De Cock's paper, by the author of the dataset, recommends removing every house of more than 4,000 sq ft before training a model: these two, and two sales at about $750,000.

## Applying what the exploration taught us

The exploration is only useful if it changes the data preparation. Apply its two findings to the training data:

- *Remove from `train_X` and `train_y` the sales of more than 4,000 sq ft of `GrLivArea`, as De Cock recommends. How many do you remove?*
- *Turn the target into its logarithm with `numpy.log1p`.*
- *Name the results `train_X_clean` and `train_y_log_clean`, then compare the correlation of `GrLivArea` with the target before and after, and plot their joint distribution again.*
- *Why don't we remove any house from `test_X`?*

**Useful documentation**

- [`pandas.Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)

In [ ]:
# Your code here

### Additional hints

- `keep = train_X["GrLivArea"] <= 4000`: a `Series` of booleans; `train_X[keep]` and `train_y[keep]` keep the same rows, and `(~keep).sum()` counts those we remove.
- `s1.corr(s2)`: the correlation of two `Series` with the same index.
- Expected: 4 sales removed; a correlation of about 0.71 before, 0.72 after.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
keep = ...
print("Sales removed:", ...)

train_X_clean = train_X[keep]
train_y_log_clean = numpy.log1p(...)

print("Before:", train_X["GrLivArea"].corr(train_y))
print("After:", train_X_clean["GrLivArea"].corr(...))

seaborn.jointplot(x=..., y=...)
plt.show()
```

### Solution

In [ ]:
keep = train_X["GrLivArea"] <= 4000
print("Sales removed:", (~keep).sum())

train_X_clean = train_X[keep]
train_y_log_clean = numpy.log1p(train_y[keep])

print(f"Correlation before: {train_X['GrLivArea'].corr(train_y):.3f}")
print(f"Correlation after: {train_X_clean['GrLivArea'].corr(train_y_log_clean):.3f}")
print(f"Skewness of the target after: {train_y_log_clean.skew():.2f}")
print("Houses of more than 4,000 sq ft in test_X:", (test_X["GrLivArea"] > 4000).sum())

seaborn.jointplot(x=train_X_clean["GrLivArea"], y=train_y_log_clean)
plt.show()

We remove 4 sales: the two partial sales, and the two sales at about $750,000, which did follow the trend (De Cock removes every house of more than 4,000 sq ft, for simplicity). The correlation of `GrLivArea` with the target goes from 0.709 to 0.719, and the skewness of the target drops to 0.07: the cloud has no isolated point left, and its trend is more regular.

We remove nothing from `test_X`: it stands for the houses we'll have to estimate in production, and we don't get to choose those. One house of more than 4,000 sq ft is in it: its prediction will be unreliable, since the model will have seen none; we can flag it to whoever uses it.